Loading initial data

In [2]:
import pandas as pd

In [3]:
data_path = "data/cleaned_gas_monitoring.csv"
df = pd.read_csv(data_path)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Time of Day                10000 non-null  str    
 1   Temperature                10000 non-null  float64
 2   Humidity                   8072 non-null   float64
 3   CO2_InfraredSensor         10000 non-null  float64
 4   CO2_ElectroChemicalSensor  10000 non-null  float64
 5   MetalOxideSensor_Unit1     10000 non-null  float64
 6   MetalOxideSensor_Unit2     8590 non-null   float64
 7   MetalOxideSensor_Unit3     10000 non-null  float64
 8   MetalOxideSensor_Unit4     10000 non-null  float64
 9   CO_GasSensor               9166 non-null   float64
 10  Session ID                 10000 non-null  int64  
 11  HVAC Operation Mode        10000 non-null  str    
 12  Ambient Light Level        8946 non-null   str    
 13  Activity Level             10000 non-null  str    
dtypes:

In [5]:
df.isnull().sum()

Time of Day                     0
Temperature                     0
Humidity                     1928
CO2_InfraredSensor              0
CO2_ElectroChemicalSensor       0
MetalOxideSensor_Unit1          0
MetalOxideSensor_Unit2       1410
MetalOxideSensor_Unit3          0
MetalOxideSensor_Unit4          0
CO_GasSensor                  834
Session ID                      0
HVAC Operation Mode             0
Ambient Light Level          1054
Activity Level                  0
dtype: int64

Missing Values, etc still remain so I'll be cleaning these to ensure its okay

In [6]:
df.duplicated().sum()

np.int64(265)

## Removing Duplicates first;

In [7]:
df_dropped = df.drop_duplicates()
df_dropped.info()
df_dropped.shape

<class 'pandas.DataFrame'>
Index: 9735 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Time of Day                9735 non-null   str    
 1   Temperature                9735 non-null   float64
 2   Humidity                   7824 non-null   float64
 3   CO2_InfraredSensor         9735 non-null   float64
 4   CO2_ElectroChemicalSensor  9735 non-null   float64
 5   MetalOxideSensor_Unit1     9735 non-null   float64
 6   MetalOxideSensor_Unit2     8333 non-null   float64
 7   MetalOxideSensor_Unit3     9735 non-null   float64
 8   MetalOxideSensor_Unit4     9735 non-null   float64
 9   CO_GasSensor               8906 non-null   float64
 10  Session ID                 9735 non-null   int64  
 11  HVAC Operation Mode        9735 non-null   str    
 12  Ambient Light Level        8685 non-null   str    
 13  Activity Level             9735 non-null   str    
dtypes: float

(9735, 14)

## Identifying numerical and categorical cols

In [8]:
numerical_features = ['Temperature', 'Humidity', 'CO2_InfraredSensor', 'CO2_ElectroChemicalSensor', 'MetalOxideSensor_Unit1', 'MetalOxideSensor_Unit2', 'MetalOxideSensor_Unit3', 'MetalOxideSensor_Unit4', 'CO_GasSensor'] # session id not included as it is not a feature for modeling
categorical_features = ['Time of Day', 'HVAC Operation Mode', 'Ambient Light Level', 'Activity Level'] # activity level to remove later because its our target variable

## Handling null values
1. Numerical columns, we using Median 
2. Categorical column (ambient light level) by choosing the light level referencing the time of day for missing data. More realistic

In [9]:
for col in ['Humidity', 'MetalOxideSensor_Unit2', 'CO_GasSensor']:
    if col in numerical_features:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

if 'Ambient Light Level' in categorical_features:
    # Impute missing light levels using the mode of their specific Time of Day
    df['Ambient Light Level'] = df.groupby('Time of Day')['Ambient Light Level'].transform(
        lambda x: x.fillna(x.mode()[0])
    )

print("\nMissing values after imputation:")
print(df[numerical_features + categorical_features].isnull().sum())


Missing values after imputation:
Temperature                  0
Humidity                     0
CO2_InfraredSensor           0
CO2_ElectroChemicalSensor    0
MetalOxideSensor_Unit1       0
MetalOxideSensor_Unit2       0
MetalOxideSensor_Unit3       0
MetalOxideSensor_Unit4       0
CO_GasSensor                 0
Time of Day                  0
HVAC Operation Mode          0
Ambient Light Level          0
Activity Level               0
dtype: int64


## Train Test Split (before fit scaling/encoding)


In [10]:
from sklearn.model_selection import train_test_split

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
activity_mapping = {
    'Low Activity': 0,
    'Moderate Activity': 1,
    'High Activity': 2
}

y = df['Activity Level'].map(activity_mapping).to_numpy()

In [ ]:
X_raw = df.drop(columns=['Activity Level', 'Session ID'])

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# Encoding

ASSUMPTION: Our problem statement ask us to find relationships. So I will encode the stuff with nominal encoding (no relationships/hierarchy), and assume nothing about them.

I will use One Hot Encoding on everything first, since it is a form of encoding that does not explictly say something has a relationship/hiererachy, etc)

But for activity level, i will add a hierarchy (because it is important)

In [ ]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder

categorical_features = ['Time of Day', 'HVAC Operation Mode', 'Ambient Light Level']
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')
X_train_cat_encoded = encoder.fit_transform(X_train_raw[categorical_features])
X_test_cat_encoded = encoder.transform(X_test_raw[categorical_features])

encoded_feature_names = encoder.get_feature_names_out(categorical_features)

X_train_cat_df = pd.DataFrame(X_train_cat_encoded, columns=encoded_feature_names, index=X_train_raw.index)
X_test_cat_df = pd.DataFrame(X_test_cat_encoded, columns=encoded_feature_names, index=X_test_raw.index)


In [ ]:
X_train_cat_df.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_raw[numerical_features])

X_test_num_scaled = scaler.transform(X_test_raw[numerical_features])

X_train_num_df = pd.DataFrame(X_train_num_scaled, columns=numerical_features, index=X_train_raw.index)
X_test_num_df = pd.DataFrame(X_test_num_scaled, columns=numerical_features, index=X_test_raw.index)

In [ ]:
X_train_num_df.head()

In [ ]:
X_train_final = pd.concat([X_train_num_df, X_train_cat_df], axis=1)
X_test_final = pd.concat([X_test_num_df, X_test_cat_df], axis=1)

In [ ]:
X_train_final.head()

In [ ]:
X_test_final.head()

In [ ]:
print(y_train[:5])
print(y_test[:5])

# Cleaning Finished
# Moving to ML

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
# 1. Define your 3 baseline models and 3 parameterized versions
models = {
    # --- MODEL 1: LOGISTIC REGRESSION ---
    "Logistic Regression (Baseline)": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "Logistic Regression (Parameterized)": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=0.1,  # Stronger L2 regularization to prevent overfitting on synthetic noise
        solver="saga",  # Efficient solver for multi-class scaled data
        random_state=42,
    ),
    # --- MODEL 2: RANDOM FOREST ---
    "Random Forest (Baseline)": RandomForestClassifier(
        class_weight="balanced", random_state=42
    ),
    "Random Forest (Parameterized)": RandomForestClassifier(
        n_estimators=200,  # More trees for a smoother decision boundary
        max_depth=10,  # Limits depth to prevent memorizing contaminated data
        min_samples_split=5,  # Requires more evidence to split a node
        class_weight="balanced_subsample",  # Adjusts weights at each tree split
        random_state=42,
    ),
    # --- MODEL 3: XGBOOST ---
    "XGBoost (Baseline)": xgb.XGBClassifier(random_state=42),
    "XGBoost (Parameterized)": xgb.XGBClassifier(
        n_estimators=150,
        max_depth=5,  # Shallower trees prevent overfitting to sensor anomalies
        learning_rate=0.05,  # Slower learning rate prevents model from converging too fast on noise
        subsample=0.8,  # Trains each tree on 80% of rows to combat synthetic contamination
        colsample_bytree=0.8,  # Trains each tree on 80% of features
        random_state=42,
    ),
}

In [ ]:
# 2. Store trained models for plotting importances later
trained_models = {}

print("training loop...\n")

for name, model in models.items():
    print(f"==================================================")
    print(f"⏳ Training: {name}...")

    # Train the model
    model.fit(X_train_final, y_train)
    trained_models[name] = model

    # Predict on the unseen validation test set
    y_pred = model.predict(X_test_final)

    # Print Evaluation Metrics
    print(f"Evaluation Report {name}:")
    # target_names maps numbers back to your human-readable activity strings
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=["Low Activity", "Moderate Activity", "High Activity"],
        )
    )

# Feature Engineering

# KV 5 fold cross validation Trial

# SMOTE

# Grid Search/ Randomized CV